# 实验二：房源采集与 MySQL 存储

- 班级：数据1241　姓名：陆禹贺垚　学号：112404260118
- 目标站点：<http://www.bspider.top/anjuke/>（安居客教学站）
- 采集字段（顺序固定）：`title → house → address → struct → area → floor → maketime → price`
- 运行环境：课程 Docker JupyterLab（Python 3.12）、课程 MySQL 8.4.11（`mysql:3306`，库 `smartpark_lab`）

> 依赖：`requests`、`beautifulsoup4`、`pymysql`。课程镜像中已安装；若缺失可在单元中运行 `%pip install requests beautifulsoup4 pymysql`。

## 一、准备：导入库与常量

`FIELDS` 是 8 个字段的固定顺序，后面解析、写 CSV、入库都复用它，避免三处顺序不一致。

In [1]:
import csv
import time
from pathlib import Path
from urllib.parse import urljoin

import bs4
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")
}
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

print("requests:", requests.__version__, " beautifulsoup4:", bs4.__version__)
print("字段顺序:", FIELDS)

requests: 2.34.2  beautifulsoup4: 4.15.0
字段顺序: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']


## 二、请求第一页并检查响应

`raise_for_status()` 负责发现 4xx/5xx；`apparent_encoding` 让中文不乱码。
注意：状态码 200 也可能返回访问验证页，所以要再打印网页标题确认是房源列表。

In [2]:
session = requests.Session()
session.headers.update(HEADERS)

resp = session.get(BASE_URL, timeout=30)
resp.raise_for_status()
resp.encoding = resp.apparent_encoding

print("状态码:", resp.status_code)
print("响应编码:", resp.encoding)
print("网页字符数:", len(resp.text))

soup = BeautifulSoup(resp.text, "html.parser")
print("网页标题:", soup.title.get_text(strip=True))

状态码: 200
响应编码: utf-8
网页字符数: 280427
网页标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客


## 三、选择器：从整页路径到“一条房源 + 相对选择器”

从标题节点向上找到**刚好包住一条房源**的节点，它的 class 是 `property`，因此用 `div.property` 选中所有房源卡片；
再在每条卡片内部用相对选择器找字段（`h3`、`.property-content-info-*`），不再依赖页面根节点的长路径。

In [3]:
rows = soup.select("div.property")
print("本页房源卡片数:", len(rows))
print("首条卡片标签/class:", rows[0].name, rows[0].get("class"))
print("首条卡片标题:", rows[0].select_one("h3").get_text(strip=True))

本页房源卡片数: 60
首条卡片标签/class: div ['property']
首条卡片标题: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


### 先打印属性组，确认顺序

面积、朝向、楼层、年代都在 `p.property-content-info-text` 里，**顺序不能想当然**。
下面把第一条房源的属性组逐项打印出来，确认第 0 项是面积、第 2 项是楼层（第 1 项是朝向，不能当楼层）。

In [4]:
row = rows[0]
info_nodes = [n for n in row.select("p.property-content-info-text")
              if "property-content-info-attribute" not in (n.get("class") or [])]
print("属性组条数:", len(info_nodes))
for i, n in enumerate(info_nodes):
    print(i, repr(n.get_text(strip=True)))

print("户型节点:", repr(row.select_one("p.property-content-info-attribute").get_text(strip=True)))
print("单价节点:", row.select_one(".property-price-average"))

属性组条数: 4
0 '89.9㎡'
1 '南北'
2 '低层(共27层)'
3 '2019年建造'
户型节点: '3室2厅1卫'
单价节点: <p class="property-price-average" data-v-53a259aa="">16465元/㎡</p>


### 解析函数：8 个字段

- 标题、楼盘、户型、单价各用 `select_one()` 取一个节点；
- 地址由多个 `<span>` 组成，直接 `get_text()` 拼接，不额外插空格；
- 面积/楼层/年代用文字特征（`㎡`、`层`、`年建造`）区分，即使顺序变化也不会把朝向当楼层；
- 单价取 `.property-price-average`（元/㎡），不是 `.property-price-total`（总价万元）。

In [5]:
def parse_row(row):
    # 从一条 div.property 中取出 8 个字段，返回 dict
    title = row.select_one("h3").get_text(strip=True)
    house = row.select_one(".property-content-info-comm-name").get_text(strip=True)
    address = row.select_one(".property-content-info-comm-address").get_text(strip=True)
    struct = row.select_one("p.property-content-info-attribute").get_text(strip=True)

    info_nodes = [n for n in row.select("p.property-content-info-text")
                  if "property-content-info-attribute" not in (n.get("class") or [])]
    texts = [n.get_text(strip=True) for n in info_nodes]

    area = next((t for t in texts if "㎡" in t), "")
    floor = next((t for t in texts if "层" in t and "年建造" not in t), "")
    maketime = next((t for t in texts if t.endswith("年建造")), "")

    price_node = row.select_one(".property-price-average")
    price = price_node.get_text(strip=True) if price_node else ""

    return {"title": title, "house": house, "address": address, "struct": struct,
            "area": area, "floor": floor, "maketime": maketime, "price": price}

### 解析第一条，逐字段对照网页

In [6]:
first = parse_row(rows[0])
for k in FIELDS:
    print(f"{k:9s}: {first[k]}")

title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    : 万科金域学府翰林菁华里
address  : 南岸学府大道二塘路59号
struct   : 3室2厅1卫
area     : 89.9㎡
floor    : 低层(共27层)
maketime : 2019年建造
price    : 16465元/㎡


### 解析整页并检查空值

面积或单价缺失时先查原因，不能直接跳过；这里统计空值，确认本页没有漏字段。

In [7]:
page_records = [parse_row(r) for r in rows]
print("本页解析条数:", len(page_records))
print("空值统计:", {k: sum(1 for rec in page_records if not rec[k]) for k in FIELDS})

本页解析条数: 60
空值统计: {'title': 0, 'house': 0, 'address': 0, 'struct': 0, 'area': 0, 'floor': 0, 'maketime': 0, 'price': 0}


## 四、下一页：href 与 urljoin

“下一页”按钮是 `a.next.next-active`，`href` 只有 `?page=2` 这样的相对地址，要用 `urljoin()` 补全；
`get_text()` 得到的是“下一页”三个字，`get("href")` 才是地址。

In [8]:
next_a = soup.select_one("a.next")
print("下一页节点:", next_a)
print("class:", next_a.get("class"))

href = next_a.get("href")
print("href 原值:", href)
print("urljoin 补全后:", urljoin(BASE_URL, href))

下一页节点: <a class="next next-active" data-npv="" data-v-29d65fc6="" href="?page=2">下一页</a>
class: ['next', 'next-active']
href 原值: ?page=2
urljoin 补全后: http://www.bspider.top/anjuke/?page=2


## 五、分页循环

停止条件按优先级判断：**到达安全页数上限 → 本页没有房源 → 找不到 `a.next` → 按钮被禁用（`next-active` 消失或 href 为 `#`）→ 越出教学站 → 地址重复**。
只有“按钮已禁用”才算正常采完末页；其余情况都记为异常中止。每页之间 `sleep(1)`，避免请求过快。

In [9]:
records = []
visited = set()
current_url = BASE_URL
page_no = 0
MAX_PAGES = 30
stop_reason = ""

while True:
    if page_no >= MAX_PAGES:
        stop_reason = f"达到页数上限 {MAX_PAGES}，异常中止（不算采集完成）"
        break

    page_no += 1
    resp = session.get(current_url, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    page_soup = BeautifulSoup(resp.text, "html.parser")

    page_rows = page_soup.select("div.property")
    if not page_rows:
        stop_reason = f"第 {page_no} 页没有解析到房源，结构异常中止"
        break

    visited.add(current_url)
    page_records = [parse_row(r) for r in page_rows]
    records.extend(page_records)
    print(f"第 {page_no:2d} 页  {current_url}  本页 {len(page_records):2d} 条  累计 {len(records)} 条")

    next_a = page_soup.select_one("a.next")
    if next_a is None:
        stop_reason = "分页区找不到 a.next，结构异常中止"
        break

    classes = next_a.get("class") or []
    href = (next_a.get("href") or "").strip()
    if "next-active" not in classes or href in ("", "#"):
        stop_reason = f"第 {page_no} 页“下一页”按钮已禁用，到达末页，正常结束"
        break

    next_url = urljoin(current_url, href)
    if not next_url.startswith(BASE_URL):
        stop_reason = f"下一页地址越出教学站：{next_url}，中止"
        break
    if next_url in visited:
        stop_reason = f"下一页地址重复：{next_url}，中止"
        break

    current_url = next_url
    time.sleep(1)

print("-" * 70)
print("停止原因:", stop_reason)
print("采集页数:", page_no)
print("原始记录数:", len(records))

第  1 页  http://www.bspider.top/anjuke/  本页 60 条  累计 60 条
第  2 页  http://www.bspider.top/anjuke/?page=2  本页 60 条  累计 120 条
第  3 页  http://www.bspider.top/anjuke/?page=3  本页 60 条  累计 180 条
第  4 页  http://www.bspider.top/anjuke/?page=4  本页 60 条  累计 240 条
第  5 页  http://www.bspider.top/anjuke/?page=5  本页 60 条  累计 300 条
第  6 页  http://www.bspider.top/anjuke/?page=6  本页 60 条  累计 360 条
第  7 页  http://www.bspider.top/anjuke/?page=7  本页 60 条  累计 420 条
第  8 页  http://www.bspider.top/anjuke/?page=8  本页 60 条  累计 480 条
第  9 页  http://www.bspider.top/anjuke/?page=9  本页 60 条  累计 540 条
第 10 页  http://www.bspider.top/anjuke/?page=10  本页 60 条  累计 600 条
----------------------------------------------------------------------
停止原因: 第 10 页“下一页”按钮已禁用，到达末页，正常结束
采集页数: 10
原始记录数: 600


## 六、去重并保存 CSV

按要求按 8 个字段完全相同去重，并记录去重前后条数；文件用 `utf-8-sig` + `newline=""`，表头就是 `FIELDS`。

In [10]:
unique_records = []
seen = set()
for rec in records:
    key = tuple(rec[f] for f in FIELDS)
    if key not in seen:
        seen.add(key)
        unique_records.append(rec)

dup_count = len(records) - len(unique_records)
print("去重前:", len(records), " 完全重复:", dup_count, " 去重后:", len(unique_records))

去重前: 600  完全重复: 102  去重后: 498


In [11]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "houseinfo.csv"

with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)
    for rec in unique_records:
        writer.writerow([rec[k] for k in FIELDS])

print("已保存:", csv_path.resolve())
print("写入数据行数:", len(unique_records))

已保存: /home/pwuser/workspace/lab02/data/houseinfo.csv
写入数据行数: 498


### 回读核对

In [12]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    csv_rows = list(reader)

print("表头:", header)
print("表头与 FIELDS 一致:", header == FIELDS)
print("回读数据行数:", len(csv_rows))
print("每条记录都是 8 项:", all(len(r) == 8 for r in csv_rows))

for i in (0, len(csv_rows) // 2, len(csv_rows) - 1):
    print(f"--- 第 {i + 1} 条 ---")
    for k in FIELDS:
        print(f"  {k:9s}: {csv_rows[i][k]}")

表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
表头与 FIELDS 一致: True
回读数据行数: 498
每条记录都是 8 项: True
--- 第 1 条 ---
  title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house    : 万科金域学府翰林菁华里
  address  : 南岸学府大道二塘路59号
  struct   : 3室2厅1卫
  area     : 89.9㎡
  floor    : 低层(共27层)
  maketime : 2019年建造
  price    : 16465元/㎡
--- 第 250 条 ---
  title    : 精装修洋房 位置安静 全新装修 随时看房
  house    : 协信城立方玺院
  address  : 沙坪坝大学城大学城南二路
  struct   : 3室3厅2卫
  area     : 95.8㎡
  floor    : 中层(共8层)
  maketime : 2016年建造
  price    : 15658元/㎡
--- 第 498 条 ---
  title    : 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
  house    : 恒大照母山山水城
  address  : 渝北照母山恒康路9号
  struct   : 3室2厅1卫
  area     : 90.2㎡
  floor    : 低层(共32层)
  maketime : 2015年建造
  price    : 15530元/㎡


## 七、写入 MySQL

先连库确认连接参数可用，再按 8 个字段整理参数，用 `executemany()` 批量写入；成功 `commit()`，出错 `rollback()`，最后关闭游标和连接。
重复运行时不靠自增 `id` 判重，而是先查出库内已有的 8 字段组合，只插入库里没有的记录。

In [13]:
import pymysql

DB_CONFIG = dict(host="mysql", port=3306, user="student", password="123456",
                 database="smartpark_lab", charset="utf8mb4",
                 autocommit=False, connect_timeout=5)

conn = pymysql.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
    print(cur.fetchone())
conn.close()

('8.4.11', 'smartpark_lab', 'student@%')


### 读取 CSV，按 8 个字段整理入库参数

`records_to_insert` 里每一项都是 8 个值的元组，顺序与 INSERT 的字段列表完全一致，正好对应 8 个 `%s`。

In [14]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    csv_rows = list(csv.DictReader(f))

records_to_insert = [tuple(row[k] for k in FIELDS) for row in csv_rows]
print("CSV 数据行数:", len(csv_rows))
print("待入库记录数:", len(records_to_insert))
print("首条参数:", records_to_insert[0])

CSV 数据行数: 498
待入库记录数: 498
首条参数: ('万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')


### 批量入库（executemany + commit / rollback / close）

In [15]:
INSERT_SQL = ("INSERT INTO houseinfo "
              "(title, house, address, struct, area, floor, maketime, price) "
              "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")

conn = None
cursor = None
try:
    conn = pymysql.connect(**DB_CONFIG)
    cursor = conn.cursor()

    cursor.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = {tuple(str(v) for v in row) for row in cursor.fetchall()}
    new_records = [r for r in records_to_insert if r not in existing]
    print("库内已有:", len(existing), " 本次待插入:", len(new_records))

    if new_records:
        affected = cursor.executemany(INSERT_SQL, new_records)
        conn.commit()
        print("executemany 影响行数:", affected, "（已 commit）")
    else:
        print("没有新记录，无需插入")

    cursor.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后条数:", cursor.fetchone()[0])

except Exception as exc:
    if conn is not None:
        conn.rollback()
        print("入库失败，已 rollback:", exc)
    raise
finally:
    if cursor is not None:
        cursor.close()
    if conn is not None:
        conn.close()
    print("已关闭游标与连接")

库内已有: 0  本次待插入: 498
executemany 影响行数: 498 （已 commit）
入库后条数: 498
已关闭游标与连接


### 重复运行校验

再次执行同一批数据：查出库内已有的 8 字段组合，应该“仍需插入 0 条”，条数不变。

In [16]:
conn = pymysql.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        before_count = cur.fetchone()[0]

        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        existing = {tuple(str(v) for v in row) for row in cur.fetchall()}
        pending = [r for r in records_to_insert if r not in existing]

        print("重跑前条数:", before_count)
        print("仍需插入:", len(pending))
        if pending:
            cur.executemany(INSERT_SQL, pending)
            conn.commit()

        cur.execute("SELECT COUNT(*) FROM houseinfo")
        print("重跑后条数:", cur.fetchone()[0])
finally:
    conn.close()

重跑前条数: 498
仍需插入: 0
重跑后条数: 498


### 抽样核对数据库中的记录

In [17]:
conn = pymysql.connect(**DB_CONFIG)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM houseinfo")
        print("库内总条数:", cur.fetchone()[0])
        cur.execute("SELECT id, title, house, address, struct, area, floor, maketime, price "
                    "FROM houseinfo ORDER BY id LIMIT 3")
        for row in cur.fetchall():
            print(row)
finally:
    conn.close()

库内总条数: 498
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')


## 八、小结

1. 页面用 `div.property` 选房源卡片，卡片内用相对选择器取 8 个字段；地址是多个 `<span>` 拼起来的，`get_text()` 直接拼接。
2. 分页靠 `a.next` 的 `href` + `urljoin()`，末页按钮失去 `next-active` 或 `href="#"` 时停止。
3. CSV 用 `utf-8-sig`、`newline=""`、`csv.writer` 保存，回读用 `csv.DictReader` 核对条数。
4. 入库读取 CSV 而不重新爬，`executemany()` 批量写入，`commit()`/`rollback()` 控制事务，重跑前先比对 8 个字段避免重复插入。